<a href="https://colab.research.google.com/github/BrendJ510/Proyectos-Escolares/blob/main/SistemasDRecomendaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
from scipy import spatial
import operator

In [ ]:
movies_df = pd.read_csv('/content/movie.csv')
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
ratings_df = pd.read_csv('/content/rating.csv')
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [ ]:
#esquema estrella, tabla de dimensiones y tabla de hechos
movies_df['name'] = movies_df.title.str[:-7]
movies_df

,movieId,title,genres,name
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II
...,...,...,...,...
27273,131254,Kein Bund für's Leben (2007),Comedy,Kein Bund für's Leben
27274,131256,"Feuer, Eis & Dosenbier (2002)",Comedy,"Feuer, Eis & Dosenbier"
27275,131258,The Pirates (2014),Adventure,The Pirates
27276,131260,Rentun Ruusu (2001),(no genres listed),Rentun Ruusu


In [ ]:
movies_df['genres'] = ([genre.split('|') for genre in movies_df['genres']])
movies_df.head()

,movieId,title,genres,name
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",Toy Story
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",Jumanji
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",Grumpier Old Men
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",Waiting to Exhale
4,5,Father of the Bride Part II (1995),[Comedy],Father of the Bride Part II


In [ ]:
gen = [] #lista vacía
for genre in movies_df['genres']:
  gen.extend(genre) #gen.append agrega listas a la lista, extend solo tengo una lista que va a agregando los demás, para quitar los duplicados

gen = list(set(gen))
gen.sort()
gen

['(no genres listed)',
 'Action',
 'Adventure',
 'Animation',
 'Children',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'IMAX',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western']

In [ ]:
gen_vec = np.zeros(len(gen), dtype = int )
gen_vec #vector de 0

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
genre = []
for genres in movies_df['genres']: #recorre filas
  gv = np.zeros(len(gen), dtype = int) #vector de 0
  for i in range(len(genres)): #recorrer todas las entradas de categorias
    for j in range(len(gen)): #vector de generos
      if genres[i] == gen[j]: # si hay coincidencia, al vector se le aumenta 1
        gv[j] += 1
        gen_vec[j] += 1
        break
  genre.append(gv)

  movies_df['genres_v'] = pd.Series(genre)
  movies_df.head()

In [ ]:
df= ratings_df.merge(movies_df, on='movieId')
df

,userId,movieId,rating,timestamp,title,genres,name,genres_v
0,1,2,3.5,2005-04-02 23:53:47,Jumanji (1995),"[Adventure, Children, Fantasy]",Jumanji,"[0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ..."
1,1,29,3.5,2005-04-02 23:31:16,"City of Lost Children, The (Cité des enfants p...","[Adventure, Drama, Fantasy, Mystery, Sci-Fi]","City of Lost Children, The (Cité des enfants p...","[0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, ..."
2,1,32,3.5,2005-04-02 23:33:39,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),"[Mystery, Sci-Fi, Thriller]",Twelve Monkeys (a.k.a. 12 Monkeys),"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
3,1,47,3.5,2005-04-02 23:32:07,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",Seven (a.k.a. Se7en),"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
4,1,50,3.5,2005-04-02 23:29:40,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]","Usual Suspects, The","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, ..."
...,...,...,...,...,...,...,...,...
65340,459,43836,1.5,2014-03-17 19:19:12,"Pink Panther, The (2006)","[Adventure, Comedy, Crime]","Pink Panther, The","[0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
65341,459,44191,4.0,2011-11-20 02:44:04,V for Vendetta (2006),"[Action, Sci-Fi, Thriller, IMAX]",V for Vendetta,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
65342,459,44195,3.0,2012-08-05 11:00:17,Thank You for Smoking (2006),"[Comedy, Drama]",Thank You for Smoking,"[0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
65343,459,44840,1.0,2013-10-21 08:43:21,"Benchwarmers, The (2006)",[Comedy],"Benchwarmers, The","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
col = ['userId', 'name','movieId','rating'
,'genres', 'genres_v']
df = df[col]
df

,userId,name,movieId,rating,genres,genres_v
0,1,Jumanji,2,3.5,"[Adventure, Children, Fantasy]","[0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ..."
1,1,"City of Lost Children, The (Cité des enfants p...",29,3.5,"[Adventure, Drama, Fantasy, Mystery, Sci-Fi]","[0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, ..."
2,1,Twelve Monkeys (a.k.a. 12 Monkeys),32,3.5,"[Mystery, Sci-Fi, Thriller]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
3,1,Seven (a.k.a. Se7en),47,3.5,"[Mystery, Thriller]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
4,1,"Usual Suspects, The",50,3.5,"[Crime, Mystery, Thriller]","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, ..."
...,...,...,...,...,...,...
65340,459,"Pink Panther, The",43836,1.5,"[Adventure, Comedy, Crime]","[0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
65341,459,V for Vendetta,44191,4.0,"[Action, Sci-Fi, Thriller, IMAX]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
65342,459,Thank You for Smoking,44195,3.0,"[Comedy, Drama]","[0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
65343,459,"Benchwarmers, The",44840,1.0,[Comedy],"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
#df.to_csv('/content/pelis.csv')

In [ ]:
#pelis_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/pelis.csv')
#pelis_df.head()
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/pelis.csv')
df.head()

,Unnamed: 0,userId,name,movieId,rating,genres,genres_v
0,0,1,Jumanji,2,3.5,"['Adventure', 'Children', 'Fantasy']",[0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
1,1,1,"City of Lost Children, The (Cité des enfants p...",29,3.5,"['Adventure', 'Drama', 'Fantasy', 'Mystery', '...",[0 0 1 0 0 0 0 0 1 1 0 0 0 0 1 0 1 0 0 0]
2,2,1,Twelve Monkeys (a.k.a. 12 Monkeys),32,3.5,"['Mystery', 'Sci-Fi', 'Thriller']",[0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0]
3,3,1,Seven (a.k.a. Se7en),47,3.5,"['Mystery', 'Thriller']",[0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0]
4,4,1,"Usual Suspects, The",50,3.5,"['Crime', 'Mystery', 'Thriller']",[0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0]


In [ ]:
ratingmovie = df["name"].value_counts()

In [ ]:
df["userId"].count()

np.int64(20000263)

In [ ]:
df['userId'].unique() #

array([     1,      2,      3, ..., 138491, 138492, 138493])

In [ ]:
df['userId'].nunique()

138493

In [ ]:

rating_m = df.groupby('name')['rating'].mean().sort_values().reset_index()
rating_m.tail(10)

,name,rating
25711,Welcome to Australia,5.0
25712,Giorgino,5.0
25713,1971,5.0
25714,Consuming Kids: The Commercialization of Child...,5.0
25715,Oranges,5.0
25716,Always for Pleasure,5.0
25717,Only Daughter,5.0
25718,When I Walk,5.0
25719,Latin Music USA,5.0
25720,Neurons to Nirvana,5.0


In [ ]:
movie_c = df['name'].value_counts().reset_index()
movie_c.head(10)

,name,count
0,Pulp Fiction,67310
1,Forrest Gump,66172
2,"Shawshank Redemption, The",63366
3,"Silence of the Lambs, The",63299
4,Jurassic Park,59715
5,Star Wars: Episode IV - A New Hope,54502
6,Braveheart,53769
7,Terminator 2: Judgment Day,52244
8,"Matrix, The",51334
9,Schindler's List,50054


In [ ]:
user_c = df['userId'].value_counts().reset_index()
user_c.head(10)

,userId,count
0,118205,9254
1,8405,7515
2,82418,5646
3,121535,5520
4,125794,5491
5,74142,5447
6,34576,5356
7,131904,5330
8,83090,5169
9,59477,4988


In [ ]:
usuarios = 1000
movie_10k = movie_c[movie_c['count']>= usuarios]
movie_10k = movie_10k.merge(rating_m, on = 'name')
movie_10k.sort_values(by ='rating', ascending = False)

,name,count,rating
2,"Shawshank Redemption, The",63366,4.446990
28,"Godfather, The",41355,4.364732
14,"Usual Suspects, The",47006,4.334372
9,Schindler's List,50054,4.310175
78,"Godfather: Part II, The",27398,4.275641
...,...,...,...
2136,Stop! Or My Mom Will Shoot,1835,1.768392
2827,Friday the 13th Part VIII: Jason Takes Manhattan,1192,1.760067
2911,Problem Child 2,1125,1.760000
2541,Baby Geniuses,1399,1.703002


In [ ]:
df_r= df[df.name.isin(list(movie_10k.name.unique()))]
df_r

,Unnamed: 0,userId,name,movieId,rating,genres,genres_v
0,0,1,Jumanji,2,3.5,"['Adventure', 'Children', 'Fantasy']",[0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
1,1,1,"City of Lost Children, The (Cité des enfants p...",29,3.5,"['Adventure', 'Drama', 'Fantasy', 'Mystery', '...",[0 0 1 0 0 0 0 0 1 1 0 0 0 0 1 0 1 0 0 0]
2,2,1,Twelve Monkeys (a.k.a. 12 Monkeys),32,3.5,"['Mystery', 'Sci-Fi', 'Thriller']",[0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0]
3,3,1,Seven (a.k.a. Se7en),47,3.5,"['Mystery', 'Thriller']",[0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0]
4,4,1,"Usual Suspects, The",50,3.5,"['Crime', 'Mystery', 'Thriller']",[0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0]
...,...,...,...,...,...,...,...
20000252,20000252,138493,WALL·E,60069,4.0,"['Adventure', 'Animation', 'Children', 'Romanc...",[0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0]
20000257,20000257,138493,X-Men Origins: Wolverine,68319,4.5,"['Action', 'Sci-Fi', 'Thriller']",[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0]
20000258,20000258,138493,Up,68954,4.5,"['Adventure', 'Animation', 'Children', 'Drama']",[0 0 1 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
20000259,20000259,138493,Transformers: Revenge of the Fallen,69526,4.5,"['Action', 'Adventure', 'Sci-Fi', 'IMAX']",[0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0]


In [ ]:
users = pd.DataFrame(df_r['userId'].value_counts().reset_index())
users.columns = ['userId','count']
users

,userId,count
0,8405,3079
1,118205,2957
2,74142,2655
3,34576,2545
4,125794,2529
...,...,...
138488,30236,6
138489,89156,6
138490,59147,5
138491,83981,5


In [ ]:
user_c = user_c.reset_index()
user_c.columns = ['userId','count']
user_c

ValueError: Length mismatch: Expected axis has 4 elements, new values have 2 elements

# SR- Base Usuario

In [ ]:
ratings = df[['userId','movieId','name','rating']]
ratings = ratings.merge(users, on = 'userId')
li = 120
ls = 650
ratings = ratings[(ratings['count']>=li)& (ratings['count'])<=ls]

NameError: name 'df' is not defined

In [ ]:
userRatings = ratings.pivot_table(index = ['userId'], columns = ['title'],
                                  values = 'rating'.reset_index)
userRatings

In [ ]:
corrMatrix = userRatings.corr(method = 'pearson',min_periods = 200) #minimo 200
corrMatrix.head()

In [ ]:
def Recomendacion_Usuario(user_id):
  #informacion de usuario
    myRatings = userRatings.loc[user_id].dropna()
    myRatings.sort_values(inplace = True, ascending = False)
    print('\n-> Usuario {} estas son algunas de las películas que más te han gustado:'.format(user_id))
    for i in range(10):
        print(" {}. {}".format(i+1, myRatings.index[i]))
    #sistema de recomendacion
    print('\n-> Estoy buscando las mejores recomendaciones para ti, no tardaré...')
    simCandidates = pd.Series(dtype = 'float64')
    #buscar usuarios similares
    for i in range(0, len(myRatings.index)):
        sims = corrMatrix[myRatings.index[i]].dropna()
        sims = sims.map(lambda x: x * myRatings[i])
        simCandidates = pd.concat([simCandidates, sims])
        #encontrar candidatos
    simCandidates.sort_values(inplace = True, ascending = False)
    simCandidates = simCandidates.groupby(simCandidates.index).sum()
    simCandidates.sort_values(inplace = True, ascending = False)
    filteredSims = simCandidates.drop(myRatings.index, errors = 'ignore')
    #recomendar peliculas
    print('\n-> Gracias por esperar, estas son unas películas que creo te pueden gustar:')
    if (len(filteredSims) < 10): #filtra una lista de 10 peliculas
      for i in range(len(filteredSims)):
        print(" {}. {}".format(i+1, filteredSims.index[i]))
    else:
      for i in range(10):
        print(" {}. {}".format(i+1, filteredSims.index[i]))

In [ ]:
user_id = input('Bienvenido!\nPor favor ingresa tu ID: ')
Recomendacion_Usuario(int(user_id))

# Recomendación systems- item based

In [ ]:
#Foto

In [ ]:
movieDict ={}
for i in range(len(moviesRatings['movieId'])):
  movieDict[moviesRatings['movieId'][i]] = (moviesRatings['name'][i],
                                            moviesRatings['genres_v'][i],
                                            moviesRatings['count'][i],
                                            moviesRatings['rating'][i],)
movieDict

In [ ]:
def Distancia(a, b):
    genresA = a[1]
    genresB = b[1]
    genreDistance = spatial.distance.cosine(genresA, genresB)
    popularityA = a[2]
    popularityB = b[2]
    popularityDistance = abs(popularityA - popularityB)
    return genreDistance + popularityDistance

In [ ]:
def Peliculas_Similares(movieID, K):
    distances = []
    for movie in movieDict:
        if (movie != movieID):
            dist = Distancia(movieDict[movieID], movieDict[movie])
            distances.append((movie, dist))
    distances.sort(key=operator.itemgetter(1))
    neighbors = []
    for x in range(K):
        neighbors.append(distances[x][0])
    return neighbors

In [ ]:
def Recomendacion_Pelicula(movie_id):
    print('\n-> ¡{} es una excelente película!'.format(movieDict[movie_id][0]))
    print('\n-> Estoy buscando peliculas similares a {}, no tardaré...'.format(movieDict[movie_id][0]))
    K = 25
    avgRating = 0
    recommendations = Peliculas_Similares(movie_id, K)
    recomendaciones = []
    for recommendation in recommendations:
        avgRating += movieDict[recommendation][3]
        recomendaciones.append([movieDict[recommendation][0], movieDict[recommendation][3]])
    rec = pd.DataFrame(recomendaciones, columns = ['Title', 'Score'])
    rec = rec.sort_values(by = ['Score']).reset_index(drop = True)
    print('\n-> Gracias por esperar, estas son unas películas que creo te pueden gustar:')
    for i in range(10):
        print(" {}. {}".format(i+1, rec['Title'][i]))

In [ ]:
movie_id = input('Bienvenido\n Por favor ingresa una película que te gusta: ')
Recomendacion_Pelicula(int(movie_id))